In [1]:
import pandas as pd
import numpy as np



In [2]:
def run_ecom_etl(input_csv_path: str) -> pd.DataFrame:
    print(f"[ETL] Initializing pipeline ingestion for: {input_csv_path}")

    df = pd.read_csv(input_csv_path)

    df = df[df['CustomerID'].notnull()]
    df = df[df['Quantity'] > 0]
    df = df[df['UnitPrice'] > 0]
    df = df.drop_duplicates()

    df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

    df['Revenue'] = df['Quantity'] * df['UnitPrice']
    df['ActivityMonth'] = df['InvoiceDate'].dt.to_period('M')

    cohort = df.groupby('CustomerID')['InvoiceDate'].min().reset_index()
    cohort.columns = ['CustomerID', 'CohortDate']
    cohort['CohortMonth'] = cohort['CohortDate'].dt.to_period('M')

    processed_df = pd.merge(
        df,
        cohort[['CustomerID', 'CohortMonth']],
        on='CustomerID'
    )

    processed_df['CohortIndex'] = (
        (processed_df['ActivityMonth'].dt.year -
         processed_df['CohortMonth'].dt.year) * 12
        +
        (processed_df['ActivityMonth'].dt.month -
         processed_df['CohortMonth'].dt.month)
    )

    print(f"[ETL] Success. Processed records: {processed_df.shape[0]}")

    return processed_df

In [3]:
input_file = r"C:\Users\chakr\Desktop\ecommerce.csv"

clean_df = run_ecom_etl(input_file)

clean_df.head()

[ETL] Initializing pipeline ingestion for: C:\Users\chakr\Desktop\ecommerce.csv
[ETL] Success. Processed records: 392691


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Revenue,ActivityMonth,CohortMonth,CohortIndex
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01,2.55,17850.0,United Kingdom,15.30,2010-12,2010-12,0
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01,3.39,17850.0,United Kingdom,20.34,2010-12,2010-12,0
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01,2.75,17850.0,United Kingdom,22.00,2010-12,2010-12,0
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01,3.39,17850.0,United Kingdom,20.34,2010-12,2010-12,0
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01,3.39,17850.0,United Kingdom,20.34,2010-12,2010-12,0
